In [11]:
import numpy as np
import scipy as sp
import sympy as smp
rng = np.random.default_rng()

In [2]:
a = []
a.append(sp.sparse.csr_matrix([1,0])) # spin 0
a.append(sp.sparse.csc_matrix([0,1])) # spin 1
Id = sp.sparse.csr_matrix(np.eye(2))
Sx = sp.sparse.csr_matrix([[0., 1.], [1., 0.]])
Sz = sp.sparse.csr_matrix([[1., 0.], [0., -1.]])
Splus = sp.sparse.csr_matrix([[0., 1.], [0., 0.]])
Sminus = sp.sparse.csr_matrix([[0., 0.], [1., 0.]])


In [70]:

def prep_sys(L):
    
    system =  rng.integers(0, 2, size=(L))
    psi = [1]
    for i in system:
        psi = sp.sparse.kron(psi,a[i])

    return psi,sp.sparse.csr_matrix(system)
def singlesite_to_full(op, i, L):
    op_list = [Id]*L  
    op_list[i] = op
    full = op_list[0]
    for op_i in op_list[1:]:
        full = sp.sparse.kron(full, op_i, format="csr")
    return full
    
def hadamard(q,l):
    
    h = sp.sparse.csr_matrix([[1,1],[1,-1]]/np.sqrt(2))
    had = singlesite_to_full(h,q,l)
    return had

def x_gate(q,l):
    x = sp.sparse.csr([0,1],[1,0])
    return singlesite_to_full(x,q,l)

def cnot(q1,q2,l):
    if q1==q2:
        raise ValueError('q1 and q2 cannto be same')
    if q1 == 0:
        return singlesite_to_full(Sx,q2,l)

    if q2 == 1:
        return 


def entropy(psi):
    l = len(psi)
    psi_ab = np.reshape(psi,(l//2,l//2))
    _,s,_ = np.linalg.svd(psi_ab)
    return -np.sum((s**2)*np.log(s**2))
    

    

In [78]:
psi, s = prep_sys(6)
s
s.shape

(1, 6)

In [79]:
psi

<1x64 sparse matrix of type '<class 'numpy.int64'>'
	with 64 stored elements (blocksize = 1x2) in Block Sparse Row format>

In [91]:
p2

<64x1 sparse matrix of type '<class 'numpy.float64'>'
	with 1 stored elements in Compressed Sparse Row format>

In [108]:
p2 = (hadamard(0,6)@p2.T).T

In [109]:
threshold=1e-12
state_to_spin_config(p2.toarray()[:,0], threshold)

[('001001', 0.3535533905932737, 0.12499999999999994),
 ('001011', -0.3535533905932737, 0.12499999999999994),
 ('011001', 0.3535533905932737, 0.12499999999999994),
 ('011011', -0.3535533905932737, 0.12499999999999994),
 ('101001', 0.3535533905932737, 0.12499999999999994),
 ('101011', -0.3535533905932737, 0.12499999999999994),
 ('111001', 0.3535533905932737, 0.12499999999999994),
 ('111011', -0.3535533905932737, 0.12499999999999994)]

In [32]:
import numpy as np

def state_to_spin_config(psi, threshold=1e-12):
    """
    Return list of (bitstring, amplitude, probability) for nonzero components of psi.
    """
    n = int(np.log2(len(psi)))
    result = []
    for i, amp in enumerate(psi):
        prob = abs(amp)**2
        if prob > threshold:
            bitstring = format(i, f'0{n}b')  # e.g. '0101'
            result.append((bitstring, amp, prob))
    return result
